In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../apps/api"))

import io
import logging
import boto3
import pandas as pd

from livewell.ingestion.ingest import _ingest_one, _s3_key
from livewell.ingestion.constants import INSTRUMENTS, INTERVALS

logging.basicConfig(level=logging.INFO)

BUCKET = "livewell-data-prod"

print(f"Setup complete. {len(INSTRUMENTS)} instruments to backfill.")

In [ ]:
print(f"{'Name':<20} {'Ticker':<15} {'s3_key'}")
print("-" * 50)
for inst in INSTRUMENTS:
    print(f"{inst['name']:<20} {inst['ticker']:<15} {inst['s3_key']}")

In [ ]:
failed = []

for inst in INSTRUMENTS:
    s3_key = inst["s3_key"]
    print(f"\n→ {inst['name']} ({s3_key}) ...", end=" ", flush=True)
    try:
        # Backfill daily only; 1h history is limited to 2 years and handled by the daily pipeline
        _ingest_one(inst, "1d", BUCKET, backfill=True)
        print("✓")
    except Exception as e:
        print(f"✗ FAILED: {e}")
        failed.append(s3_key)

print("\n" + "=" * 50)
if failed:
    print(f"FAILED ({len(failed)}): {', '.join(failed)}")
else:
    print(f"All {len(INSTRUMENTS)} instruments backfilled successfully.")

In [ ]:
s3 = boto3.client("s3")

print(f"{'Instrument':<12} {'Files':<8} {'Min Date':<14} {'Max Date':<14} {'Rows'}")
print("-" * 60)

for inst in INSTRUMENTS:
    s3_key = inst["s3_key"]
    prefix = f"prices/{s3_key}/1d/"
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    assert not resp.get("IsTruncated"), f"S3 listing truncated for {prefix} — add pagination"
    keys = [o["Key"] for o in resp.get("Contents", [])]
    if not keys:
        print(f"{s3_key:<12} {'NO DATA'}")
        continue

    frames = []
    for key in sorted(keys):
        try:
            obj = s3.get_object(Bucket=BUCKET, Key=key)
            df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
            frames.append(df)
        except Exception as e:
            print(f"  WARNING: could not read {key}: {e}")

    if not frames:
        print(f"{s3_key:<12} {'READ FAILED'}")
        continue

    combined = pd.concat(frames, ignore_index=True)
    combined["date"] = pd.to_datetime(combined["date"])
    min_date = combined["date"].min().strftime("%Y-%m-%d")
    max_date = combined["date"].max().strftime("%Y-%m-%d")
    print(f"{s3_key:<12} {len(keys):<8} {min_date:<14} {max_date:<14} {len(combined)}")